In [0]:
spark.conf.set(
  "fs.azure.account.key.salesviewacc.dfs.core.windows.net",
  "<accesskey>"
)

In [0]:
df = spark.read.csv(
  "abfss://bronze@salesviewacc.dfs.core.windows.net/sales_view/store",
  header=True,
  inferSchema=True
)


In [0]:
import re

def to_snake_case(col):
    col = col.strip()
    col = re.sub(r'[^\w]+', '_', col)
    col = re.sub(r'(?<!^)(?=[A-Z])', '_', col).lower()
    return col

In [0]:
df = df.toDF(*[to_snake_case(c) for c in df.columns])
print(df.columns)


In [0]:
from pyspark.sql.functions import to_date

df = df.withColumn("created_at", to_date(col("created_at"))) \
       .withColumn("updated_at", to_date(col("updated_at")))


In [0]:
from pyspark.sql.functions import split , col

df = df.withColumn("store_category",
        split(col("email_address"), "@")[1].substr(1, 100))


In [0]:
print(df.columns)

In [0]:
df.write.format("delta") \
  .mode("overwrite") \
  .save("abfss://silver@salesviewacc.dfs.core.windows.net/sales_view/store")
